In [170]:
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.cluster import AgglomerativeClustering
from anytree import Node, RenderTree

warnings.filterwarnings("ignore")

In [172]:
df = pd.read_json("D:/git/Taxonomy_Buidling_Textual_Corpora/data/icecat_data_train.json")

# Work only with first 5000 rows from the beginning
df = df.iloc[:5000].copy()

print("Raw sample size:", len(df))


Raw sample size: 5000


In [173]:
def split_path(path):
    if not isinstance(path, str):
        return []
    parts = [p.strip() for p in path.split(">") if p.strip()]
    return parts

df["num_levels"] = df["pathlist_names"].apply(lambda x: len(split_path(x)))

print("Counts BEFORE cleaning:")
print(df["num_levels"].value_counts().sort_index())

Counts BEFORE cleaning:
num_levels
3    2883
4    2117
Name: count, dtype: int64


In [174]:
import pandas as pd

def make_3_and_4(path):
    parts = split_path(path)

    # default
    path_3 = None
    level_4 = None

    if len(parts) >= 3:
        path_3 = " > ".join(parts[:3])   # A > B > C
    if len(parts) >= 4:
        level_4 = parts[3]               # D (4th level)

    return pd.Series({"path_3": path_3, "level_4": level_4})

df[["path_3", "level_4"]] = df["pathlist_names"].apply(make_3_and_4)

print(df[["pathlist_names", "path_3", "level_4"]].head(10))


                                            pathlist_names  \
1072689  Computers & Electronics>Computers>PCs/Workstat...   
906402   Computers & Electronics>Computers>Notebook Par...   
411281   Computers & Electronics>Computer Cables>Fibre ...   
425903   Computers & Electronics>Computers>Handheld Mob...   
1047582  Computers & Electronics>Computers>PCs/Workstat...   
904910   Computers & Electronics>Computers>Notebook Par...   
157385   Computers & Electronics>Software>Software Lice...   
934548   Computers & Electronics>Computers>Notebook Par...   
876762   Computers & Electronics>Computers>Notebook Par...   
397028   Computers & Electronics>Telecom & Navigation>M...   

                                                    path_3  \
1072689  Computers & Electronics > Computers > PCs/Work...   
906402   Computers & Electronics > Computers > Notebook...   
411281   Computers & Electronics > Computer Cables > Fi...   
425903   Computers & Electronics > Computers > Handheld...   
1047582

In [175]:
print("Rows:", len(df))
print("Unique num_levels (original):")
print(df["num_levels"].value_counts().sort_index())

print("\nCheck how many have a 4th level stored:")
print(df["level_4"].notna().sum(), "rows with level_4")


Rows: 5000
Unique num_levels (original):
num_levels
3    2883
4    2117
Name: count, dtype: int64

Check how many have a 4th level stored:
2117 rows with level_4


In [176]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 1072689 to 1121736
Data columns (total 48 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Brand                                       5000 non-null   object 
 1   BrandInfo.BrandLocalName                    5000 non-null   object 
 2   BrandInfo.BrandLogo                         4994 non-null   object 
 3   BrandInfo.BrandName                         5000 non-null   object 
 4   BrandLogo                                   4994 non-null   object 
 5   BrandPartCode                               5000 non-null   object 
 6   BulletPoints                                4654 non-null   object 
 7   Category.CategoryID                         5000 non-null   int64  
 8   Category.Name.Language                      5000 non-null   object 
 9   Category.Name.Value                         5000 non-null   object 
 10  Descript

In [177]:
# Extract A, B, C from path_3
df[["A", "B", "C"]] = (
    df["path_3"]
    .str.split(">", expand=True)
    .apply(lambda col: col.str.strip())
)


In [178]:
df.head()

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names,num_levels,path_3,level_4,A,B,C
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...,4,Computers & Electronics > Computers > Notebook...,Notebook Spare Parts,Computers & Electronics,Computers,Notebook Parts & Accessories
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...,3,Computers & Electronics > Computer Cables > Fi...,None,Computers & Electronics,Computer Cables,Fibre Optic Cables
425903,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,FA889AA#AC3,[],8194,EN,Handheld Mobile Computer Spare Parts,...,None,NaN,2833>150>8194,Computers & Electronics>Computers>Handheld Mob...,3,Computers & Electronics > Computers > Handheld...,None,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations


In [179]:
# Basic Stats
print("Total products:", len(df))
print("Unique A-level:", df["A"].nunique())
print("Unique B-level:", df["B"].nunique())
print("Unique C-level:", df["C"].nunique())


Total products: 5000
Unique A-level: 1
Unique B-level: 17
Unique C-level: 160


In [180]:
#Distribution of A / B / C/
print("\nProducts per A-level:")
print(df["A"].value_counts())

print("\nProducts per B-level (top 10):")
print(df["B"].value_counts().head(20))

print("\nProducts per C-level (top 10):")
print(df["C"].value_counts().head(20))

print("\nProducts per C-level (bottom 10):")
print(df["C"].value_counts().tail(10))



Products per A-level:
A
Computers & Electronics    5000
Name: count, dtype: int64

Products per B-level (top 10):
B
Computers                           2202
Printers & Scanners                  368
Computer Components                  330
Warranty & Support                   318
Software                             296
Data Storage                         284
TVs & Monitors                       251
Computer Cables                      205
Telecom & Navigation                 189
Batteries & Power Supplies           141
Data Input Devices                   111
Consumer Audio & Video Equipment     110
Projectors                            73
Photo & Video Equipment               66
Networking                            47
Office Electronics                     7
Smart Wearables                        2
Name: count, dtype: int64

Products per C-level (top 10):
C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage

In [181]:
# C-level Imbalance Quantiles
# (This shows how skewed leaf categories are)
prod_per_C = df["C"].value_counts()

print("\nC-level product quantiles:")
print(prod_per_C.quantile([0.1, 0.25, 0.5, 0.75, 0.9]))



C-level product quantiles:
0.10     1.0
0.25     2.0
0.50     5.0
0.75    14.0
0.90    60.3
Name: count, dtype: float64


In [182]:
# Basic counts
print("Total products:", len(df))
print("Unique A levels:", df["A"].nunique())
print("Unique B levels:", df["B"].nunique())
print("Unique C levels:", df["C"].nunique())

# Compute product counts
prod_A = df["A"].value_counts()
prod_B = df["B"].value_counts()
prod_C = df["C"].value_counts()


Total products: 5000
Unique A levels: 1
Unique B levels: 17
Unique C levels: 160


In [183]:
# Identify problematic A/B/C categories
# Very large categories (top 10)
print("\n🔥 TOP 10 LARGEST C categories:")
print(prod_C.head(10))

# Very small categories (bottom 10)
print("\n❗ BOTTOM 10 SMALLEST C categories:")
print(prod_C.tail(10))

# C categories with < 10 products
weak_C = prod_C[prod_C < 10]
print("\n❗ Weak C categories (<10 products):", len(weak_C))
print(weak_C.head(20))

# B categories with < 3 C children
C_children_per_B = df.groupby("B")["C"].nunique()
weak_B = C_children_per_B[C_children_per_B < 3]

print("\n❗ Weak B categories (<3 C children):", len(weak_B))
print(weak_B.head(20))



🔥 TOP 10 LARGEST C categories:
C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage Devices              273
System Components                 241
PCs/Workstations                  227
Software Licenses/Upgrades        227
Printing Supplies                 215
Keyboards                          85
Chassis Components                 84
Name: count, dtype: int64

❗ BOTTOM 10 SMALLEST C categories:
C
SCART Cables                                         1
Fax Machines                                         1
Sport Watch Accessories                              1
Network Switch Components                            1
Uninterruptible Power Supplies (UPSs) Accessories    1
USB Graphics Adapters                                1
Two-Way Radios                                       1
UPS Battery Cabinets                                 1
AV Receivers                                         1
S-Video Cables             

In [184]:
#Identify dominant categories (too large)
# Very large C categories (skewing the distribution)
dominant_C = prod_C[prod_C > prod_C.quantile(0.95)]

print("\n🔥 Dominant C categories (>95th percentile):", len(dominant_C))
print(dominant_C)



🔥 Dominant C categories (>95th percentile): 8
C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage Devices              273
System Components                 241
PCs/Workstations                  227
Software Licenses/Upgrades        227
Printing Supplies                 215
Name: count, dtype: int64


In [185]:
B_summary = (
    df.groupby(["A", "B"])
      .agg(num_products=("C", "count"),
           num_C=("C", "nunique"))
      .reset_index()
      .sort_values("num_products", ascending=False)
)

print(" B-level imbalance summary:")
print(B_summary.head(20))


 B-level imbalance summary:
                          A                                 B  num_products  \
3   Computers & Electronics                         Computers          2202   
10  Computers & Electronics               Printers & Scanners           368   
2   Computers & Electronics               Computer Components           330   
16  Computers & Electronics                Warranty & Support           318   
13  Computers & Electronics                          Software           296   
6   Computers & Electronics                      Data Storage           284   
14  Computers & Electronics                    TVs & Monitors           251   
1   Computers & Electronics                   Computer Cables           205   
15  Computers & Electronics              Telecom & Navigation           189   
0   Computers & Electronics        Batteries & Power Supplies           141   
5   Computers & Electronics                Data Input Devices           111   
4   Computers & Electron

In [186]:
def report_AB(df):
    grouped = (
        df.groupby(["A", "B"])
          .agg(
              num_C=("C", "nunique"),
              num_products=("C", "size")
          )
          .reset_index()
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        temp = grouped[grouped["A"] == a]
        for _, row in temp.iterrows():
            print(f"  B: {row['B']}")
            print(f"    #C-level subcategories: {row['num_C']}")
            print(f"    #products (total under this B): {row['num_products']}")


In [187]:
report_AB(df)


A: Computers & Electronics
  B: Batteries & Power Supplies
    #C-level subcategories: 12
    #products (total under this B): 141
  B: Computer Cables
    #C-level subcategories: 23
    #products (total under this B): 205
  B: Computer Components
    #C-level subcategories: 3
    #products (total under this B): 330
  B: Computers
    #C-level subcategories: 18
    #products (total under this B): 2202
  B: Consumer Audio & Video Equipment
    #C-level subcategories: 26
    #products (total under this B): 110
  B: Data Input Devices
    #C-level subcategories: 3
    #products (total under this B): 111
  B: Data Storage
    #C-level subcategories: 2
    #products (total under this B): 284
  B: Networking
    #C-level subcategories: 13
    #products (total under this B): 47
  B: Office Electronics
    #C-level subcategories: 3
    #products (total under this B): 7
  B: Photo & Video Equipment
    #C-level subcategories: 3
    #products (total under this B): 66
  B: Printers & Scanners
   

In [188]:
def report_ABC(df):
    grouped = (
        df.groupby(["A", "B", "C"])
          .size()
          .reset_index(name="num_products")
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        tempA = grouped[grouped["A"] == a]

        for b in tempA["B"].unique():
            print(f"  B: {b}")
            tempB = tempA[tempA["B"] == b]

            for _, row in tempB.iterrows():
                print(f"    C: {row['C']}  ({row['num_products']} products)")


In [189]:
report_ABC(df)



A: Computers & Electronics
  B: Batteries & Power Supplies
    C: Battery Chargers  (4 products)
    C: Household Batteries  (11 products)
    C: Notebook Power Tips  (2 products)
    C: Portable Device Management Carts & Cabinets  (1 products)
    C: Power Adapters & Inverters  (63 products)
    C: Power Banks  (4 products)
    C: Power Distribution Units (PDUs)  (11 products)
    C: Power Supply Units  (16 products)
    C: UPS Batteries  (3 products)
    C: UPS Battery Cabinets  (1 products)
    C: Uninterruptible Power Supplies (UPSs)  (24 products)
    C: Uninterruptible Power Supplies (UPSs) Accessories  (1 products)
  B: Computer Cables
    C: Audio Cables  (3 products)
    C: Cable Interface/Gender Adapters  (20 products)
    C: Cable Protectors  (2 products)
    C: Coaxial Cables  (4 products)
    C: DVI Cables  (5 products)
    C: DisplayPort Cables  (3 products)
    C: Fibre Optic Cables  (29 products)
    C: FireWire Cables  (1 products)
    C: HDMI Cables  (11 products)
  

In [190]:
df.head()

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names,num_levels,path_3,level_4,A,B,C
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...,4,Computers & Electronics > Computers > Notebook...,Notebook Spare Parts,Computers & Electronics,Computers,Notebook Parts & Accessories
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...,3,Computers & Electronics > Computer Cables > Fi...,None,Computers & Electronics,Computer Cables,Fibre Optic Cables
425903,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,FA889AA#AC3,[],8194,EN,Handheld Mobile Computer Spare Parts,...,None,NaN,2833>150>8194,Computers & Electronics>Computers>Handheld Mob...,3,Computers & Electronics > Computers > Handheld...,None,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,3,Computers & Electronics > Computers > PCs/Work...,None,Computers & Electronics,Computers,PCs/Workstations


In [191]:
df[['ProductName',"BrandPartCode"]]

,ProductName,BrandPartCode
1072689,K31CD-IT049T,K31CD-IT049T
906402,686915-A41,686915-A41
411281,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,37745
425903,FA889AA,FA889AA#AC3
1047582,C30,109559U
...,...,...
231002,490-BDZR,490-BDZR
978436,SIC1094057LCD0,SIC1094057LCD0
1101608,Elite Slice G2 with Microsoft Teams Rooms,6GV21PA
66861,T300,T300.ADEUPK


In [192]:
def report_ABC_products(df):
    for a in df["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        dfA = df[df["A"] == a]

        for b in dfA["B"].unique():
            print(f"  B: {b}")
            dfB = dfA[dfA["B"] == b]

            for c in dfB["C"].unique():
                dfC = dfB[dfB["C"] == c]
                print(f"    C: {c}  ({len(dfC)} products)")
                
                # show top 10 product codes
                for code in dfC["ProductName"].fillna("").head(10):
                    print(f"        • {code}")


In [193]:
report_ABC_products(df)



A: Computers & Electronics
  B: Computers
    C: PCs/Workstations  (227 products)
        • K31CD-IT049T
        • C30
        • UN42-M031M
        • P310
        • 875-1303ng
        • 400 G6
        • M910q
        • 3 VR7RD-037US
        • 400 G4 + EliteDisplay E223
        • ProDesk 600 G3 Desktop Mini PC
    C: Notebook Parts & Accessories  (1019 products)
        • 686915-A41
        • 659501-BB1
        • 00HT024
        • 448002-001
        • Top Cover & Keyboard (Italy)
        • 25212106
        • SIC1085866LCD0
        • P000453890
        • SIC1099842LCD0
        • 25207264
    C: Handheld Mobile Computer Spare Parts  (1 products)
        • FA889AA
    C: Notebooks  (774 products)
        • V3-371-35U2
        • GE72 2QC(Apache)-209NL
        • P2510-M-54EH
        • GF72 8RE-062X
        • 15-ay045ns
        • GX531GX-ES016T
        • 460
        • 840 G5
        • R30-A-17G
        • UX310UF-FC002T
    C: Servers  (15 products)
        • x3550 M5
        • RD230
        

In [194]:
# Compute metrics
products_per_AB = df.groupby(["A", "B"]).size()
C_children_per_AB = df.groupby(["A", "B"])["C"].nunique()

# Combine
ab_summary = pd.DataFrame({
    "products_in_AB": products_per_AB,
    "n_C_children": C_children_per_AB
})

# Sort by product count descending (like your example)
ab_summary = ab_summary.sort_values("products_in_AB", ascending=False).head(10)

print(ab_summary)


                                                    products_in_AB  \
A                       B                                            
Computers & Electronics Computers                             2202   
                        Printers & Scanners                    368   
                        Computer Components                    330   
                        Warranty & Support                     318   
                        Software                               296   
                        Data Storage                           284   
                        TVs & Monitors                         251   
                        Computer Cables                        205   
                        Telecom & Navigation                   189   
                        Batteries & Power Supplies             141   

                                                    n_C_children  
A                       B                                         
Computers & Electronics C

In [195]:
prod_per_C = df["C"].value_counts()
prod_per_C


C
Notebook Parts & Accessories     1019
Notebooks                         774
Warranty & Support Extensions     288
Data Storage Devices              273
System Components                 241
                                 ... 
USB Graphics Adapters               1
Two-Way Radios                      1
UPS Battery Cabinets                1
AV Receivers                        1
S-Video Cables                      1
Name: count, Length: 160, dtype: int64

In [196]:
#Keep only C categories with > 20 products
valid_C = prod_per_C[prod_per_C >= 20].index
df = df[df["C"].isin(valid_C)].copy()


In [197]:
print("Remaining rows:", len(df))
print("Unique C after filtering:", df["C"].nunique())
print("Min products per C (should be >20):", df["C"].value_counts().min())


Remaining rows: 4382
Unique C after filtering: 32
Min products per C (should be >20): 20


In [198]:
#final A-B summary
def print_final_AB(df):
    grouped = (
        df.groupby(["A", "B"])
          .agg(
              num_C_under_B=("C", "nunique"),
              num_products_under_B=("C", "size")
          )
          .reset_index()
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        temp = grouped[grouped["A"] == a]

        for _, row in temp.iterrows():
            print(f"  B: {row['B']}")
            print(f"    #C-level subcategories: {row['num_C_under_B']}")
            print(f"    #products (under this B): {row['num_products_under_B']}")


In [199]:
print_final_AB(df)



A: Computers & Electronics
  B: Batteries & Power Supplies
    #C-level subcategories: 2
    #products (under this B): 87
  B: Computer Cables
    #C-level subcategories: 3
    #products (under this B): 116
  B: Computer Components
    #C-level subcategories: 2
    #products (under this B): 325
  B: Computers
    #C-level subcategories: 6
    #products (under this B): 2144
  B: Consumer Audio & Video Equipment
    #C-level subcategories: 1
    #products (under this B): 23
  B: Data Input Devices
    #C-level subcategories: 1
    #products (under this B): 85
  B: Data Storage
    #C-level subcategories: 1
    #products (under this B): 273
  B: Photo & Video Equipment
    #C-level subcategories: 1
    #products (under this B): 39
  B: Printers & Scanners
    #C-level subcategories: 4
    #products (under this B): 363
  B: Projectors
    #C-level subcategories: 1
    #products (under this B): 25
  B: Software
    #C-level subcategories: 2
    #products (under this B): 280
  B: TVs & Moni

In [200]:
def print_final_ABC(df):
    grouped = (
        df.groupby(["A", "B", "C"])
          .size()
          .reset_index(name="num_products_under_C")
    )

    for a in grouped["A"].unique():
        print(f"\nA: {a}")
        print("="*80)
        tempA = grouped[grouped["A"] == a]

        for b in tempA["B"].unique():
            print(f"  B: {b}")
            tempB = tempA[tempA["B"] == b]

            for _, row in tempB.iterrows():
                print(f"    C: {row['C']}  ({row['num_products_under_C']} products)")
                
print_final_ABC(df)




A: Computers & Electronics
  B: Batteries & Power Supplies
    C: Power Adapters & Inverters  (63 products)
    C: Uninterruptible Power Supplies (UPSs)  (24 products)
  B: Computer Cables
    C: Cable Interface/Gender Adapters  (20 products)
    C: Fibre Optic Cables  (29 products)
    C: Networking Cables  (67 products)
  B: Computer Components
    C: Chassis Components  (84 products)
    C: System Components  (241 products)
  B: Computers
    C: All-in-One PCs/Workstations  (60 products)
    C: Notebook Parts & Accessories  (1019 products)
    C: Notebooks  (774 products)
    C: PCs/Workstations  (227 products)
    C: Tablet Cases  (25 products)
    C: Tablets  (39 products)
  B: Consumer Audio & Video Equipment
    C: Audio Equipment Parts & Accessories  (23 products)
  B: Data Input Devices
    C: Keyboards  (85 products)
  B: Data Storage
    C: Data Storage Devices  (273 products)
  B: Photo & Video Equipment
    C: Cameras & Camcorders  (39 products)
  B: Printers & Scanners
 

In [201]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4382 entries, 1072689 to 1121736
Data columns (total 51 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Brand                                       4382 non-null   object 
 1   BrandInfo.BrandLocalName                    4382 non-null   object 
 2   BrandInfo.BrandLogo                         4379 non-null   object 
 3   BrandInfo.BrandName                         4382 non-null   object 
 4   BrandLogo                                   4379 non-null   object 
 5   BrandPartCode                               4382 non-null   object 
 6   BulletPoints                                4088 non-null   object 
 7   Category.CategoryID                         4382 non-null   int64  
 8   Category.Name.Language                      4382 non-null   object 
 9   Category.Name.Value                         4382 non-null   object 
 10  Descript

In [202]:
#columns to keep
cols_needed = [
    "Brand",
    "BrandPartCode",
    "ProductName",
    "Description.LongProductName",
    "SummaryDescription.LongSummaryDescription",
    "Description.LongDesc",
    "A", "B", "C",
    "path_3"   # <-- Needed for evaluation
]


In [203]:
df_clean = df[cols_needed].copy()


In [204]:
# Build raw metadata text

def build_metadata(row):
    parts = []

    if pd.notna(row["Brand"]):
        parts.append(row["Brand"])

    if pd.notna(row["BrandPartCode"]):
        parts.append(row["BrandPartCode"])
        
    if pd.notna(row["ProductName"]):
        parts.append(row["ProductName"])

    if pd.notna(row["Description.LongProductName"]):
        parts.append(row["Description.LongProductName"])

    if pd.notna(row["SummaryDescription.LongSummaryDescription"]):
        parts.append(row["SummaryDescription.LongSummaryDescription"])

    if pd.notna(row["Description.LongDesc"]):
        parts.append(row["Description.LongDesc"])

    return " ".join(parts)

df_clean["metadata_text"] = df_clean.apply(build_metadata, axis=1)


In [205]:
df_clean.head(3)

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...


In [206]:
import re

def clean_text(t):
    if not isinstance(t, str):
        return ""

    t = re.sub(r"<[^>]+>", " ", t)          # remove HTML tags
    t = re.sub(r"\s+", " ", t)              # collapse spaces
    t = t.replace("\xa0", " ")              # remove non-breaking
    t = t.strip()

    return t

df_clean["metadata_text_clean"] = df_clean["metadata_text"].apply(clean_text)


In [207]:
df_clean.head()

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...
1047582,Lenovo,109559U,C30,"Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",Lenovo ThinkStation C30. Processor frequency: ...,The C30 builds on its award-winning design as ...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...
904910,HP,659501-BB1,659501-BB1,Keyboard in ash black for use in Israel (inclu...,HP 659501-BB1. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,HP 659501-BB1 659501-BB1 Keyboard in ash black...


In [208]:
#check the average length of metadata_clean_text
df_clean["meta_len"] = df_clean["metadata_text_clean"].apply(lambda x: len(str(x)))

print(df_clean["meta_len"].describe())


count     4382.000000
mean      1318.927887
std       1831.572922
min         29.000000
25%        236.000000
50%        626.000000
75%       1759.750000
max      27361.000000
Name: meta_len, dtype: float64


In [209]:
# This creates a new column telling YAKE exactly how many keywords to extract for each product.
def decide_top_k(meta_len):
    if meta_len < 150:
        return 5
    elif meta_len < 500:
        return 10
    elif meta_len < 1500:
        return 15
    elif meta_len < 3000:
        return 20
    elif meta_len < 8000:
        return 30
    else:
        return 50

df_clean["yake_k"] = df_clean["meta_len"].apply(decide_top_k)


In [210]:
df_clean.head()

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,meta_len,yake_k
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,3657,30
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...,299,10
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,1840,20
1047582,Lenovo,109559U,C30,"Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",Lenovo ThinkStation C30. Processor frequency: ...,The C30 builds on its award-winning design as ...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,2905,20
904910,HP,659501-BB1,659501-BB1,Keyboard in ash black for use in Israel (inclu...,HP 659501-BB1. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,201,10


In [211]:
#Extract YAKE keywords per row
import yake

kw_extractor = yake.KeywordExtractor(
    n=1,             # extract unigrams
    dedupLim=0.9,    # reduce duplicates
    dedupFunc="seq"
)

def extract_yake(text, k):
    try:
        kws = kw_extractor.extract_keywords(text)
        return [kw for kw, score in kws[:k]]
    except:
        return []

df_clean["yake_keywords"] = df_clean.apply(
    lambda row: extract_yake(row["metadata_text_clean"], row["yake_k"]),
    axis=1
)


In [212]:
df_clean.head()

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,meta_len,yake_k,yake_keywords
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,3657,30,"[ASUS, Intel, Windows, Core, Graphics, Cache, ..."
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...,299,10,"[Keyboard, Belgium, backlight, cable, includes..."
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,1840,20,"[Duplex, Blue, Fiber, Single-Mode, Patch, Plen..."
1047582,Lenovo,109559U,C30,"Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",Lenovo ThinkStation C30. Processor frequency: ...,The C30 builds on its award-winning design as ...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,2905,20,"[Lenovo, Intel, Xeon, Cache, Windows, Professi..."
904910,HP,659501-BB1,659501-BB1,Keyboard in ash black for use in Israel (inclu...,HP 659501-BB1. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,201,10,"[Israel, Keyboard, includes, cable, compatibil..."


In [213]:
def yake_list_to_text(kws):
    if isinstance(kws, list):
        # join phrases with space; or use ", ".join(kws) if you prefer
        return " ".join(str(k).strip() for k in kws if isinstance(k, str))
    elif pd.isna(kws):
        return ""
    else:
        return str(kws)

df_clean["yake_text"] = df_clean["yake_keywords"].apply(yake_list_to_text)
df_clean[["metadata_text_clean", "yake_keywords", "yake_text"]].head(3)

# metadata_text_clean → full raw description (for comparison / ablation later)
# yake_keywords → list of extracted phrases
# yake_text → compact keyword string we’ll embed

,metadata_text_clean,yake_keywords,yake_text
1072689,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,"[ASUS, Intel, Windows, Core, Graphics, Cache, ...",ASUS Intel Windows Core Graphics Cache RAM Eth...
906402,HP 686915-A41 686915-A41 Keyboard in midnight ...,"[Keyboard, Belgium, backlight, cable, includes...",Keyboard Belgium backlight cable includes comp...
411281,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,"[Duplex, Blue, Fiber, Single-Mode, Patch, Plen...",Duplex Blue Fiber Single-Mode Patch Plenum-Rat...


In [214]:
df_clean.head(2)

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,meta_len,yake_k,yake_keywords,yake_text
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,3657,30,"[ASUS, Intel, Windows, Core, Graphics, Cache, ...",ASUS Intel Windows Core Graphics Cache RAM Eth...
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...,299,10,"[Keyboard, Belgium, backlight, cable, includes...",Keyboard Belgium backlight cable includes comp...


In [215]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1) Load a light, fast model (same family we used before)
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2) Take the YAKE-based keyword text
texts_for_embed = df_clean["yake_text"].fillna("").tolist()

# 3) Encode into embeddings (L2-normalized)
embeddings = st_model.encode(
    texts_for_embed,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings.shape


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

(4382, 384)

In [216]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=15,        # local structure
    min_dist=0.0,          # tight clusters
    n_components=25,       # richer density structure
    metric="cosine",       # BEST for normalized embeddings
    random_state=42
)

reduced = umap_model.fit_transform(embeddings)
reduced.shape


(4382, 25)

In [217]:
import hdbscan

clusterer_C = hdbscan.HDBSCAN(
    min_cluster_size=20,   # matches your filtered C ≥ 20 idea
    min_samples=1,         # allow fine-grained clusters
    metric="euclidean",    # MUST BE euclidean after UMAP
    cluster_selection_method="eom"
)

df_clean["C_id"] = clusterer_C.fit_predict(reduced)
df_clean["C_id"].value_counts().head(20)


C_id
-1     380
 9     354
 45    190
 21    123
 38    113
 5      90
 3      88
 8      85
 27     85
 16     84
 62     77
 41     77
 77     75
 76     75
 44     73
 79     72
 66     67
 51     65
 17     63
 31     63
Name: count, dtype: int64

In [218]:
# #  Drop HDBSCAN noise before naming
# df_clean = df_clean[df_clean["C_id"] != -1].copy()
# #removing noise

In [219]:
df_clean

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,meta_len,yake_k,yake_keywords,yake_text,C_id
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,3657,30,"[ASUS, Intel, Windows, Core, Graphics, Cache, ...",ASUS Intel Windows Core Graphics Cache RAM Eth...,66
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...,299,10,"[Keyboard, Belgium, backlight, cable, includes...",Keyboard Belgium backlight cable includes comp...,49
411281,C2G,37745,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,Get the performance you demand at a price that...,Computers & Electronics,Computer Cables,Fibre Optic Cables,Computers & Electronics > Computer Cables > Fi...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,C2G 37745 1m ST/SC Plenum-Rated 9/125 Duplex S...,1840,20,"[Duplex, Blue, Fiber, Single-Mode, Patch, Plen...",Duplex Blue Fiber Single-Mode Patch Plenum-Rat...,48
1047582,Lenovo,109559U,C30,"Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",Lenovo ThinkStation C30. Processor frequency: ...,The C30 builds on its award-winning design as ...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,Lenovo 109559U C30 Intel Xeon E5-2620 (15M Cac...,2905,20,"[Lenovo, Intel, Xeon, Cache, Windows, Professi...",Lenovo Intel Xeon Cache Windows Professional Q...,77
904910,HP,659501-BB1,659501-BB1,Keyboard in ash black for use in Israel (inclu...,HP 659501-BB1. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,HP 659501-BB1 659501-BB1 Keyboard in ash black...,201,10,"[Israel, Keyboard, includes, cable, compatibil...",Israel Keyboard includes cable compatibility H...,49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231002,DELL,490-BDZR,490-BDZR,Radeon Pro WX 2100 2 GB 1 DP 2 mDP (Precision),"DELL 490-BDZR. Graphics processor family: AMD,...",The new Radeon™ Pro WX 2100 graphics card is t...,Computers & Electronics,Computer Components,System Components,Computers & Electronics > Computer Components ...,DELL 490-BDZR 490-BDZR Radeon Pro WX 2100 2 GB...,DELL 490-BDZR 490-BDZR Radeon Pro WX 2100 2 GB...,765,15,"[DELL, Pro, Precision, Radeon, Graphics, memor...",DELL Pro Precision Radeon Graphics memory mDP ...,22
978436,Toshiba,SIC1094057LCD0,SIC1094057LCD0,Satellite A660 15.6 replacement laptop LCD screen,Toshiba SIC1094057LCD0. Type: Display. Display...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,Toshiba SIC1094057LCD0 SIC1094057LCD0 Satellit...,Toshiba SIC1094057LCD0 SIC1094057LCD0 Satellit...,221,10,"[Satellite, LCD, Toshiba, Display, replacement...",Satellite LCD Toshiba Display replacement comp...,3
1101608,HP,6GV21PA,Elite Slice G2 with Microsoft Teams Rooms,None,HP Elite Slice G2 with Microsoft Teams Rooms,None,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,HP 6GV21PA Elite Slice G2 with Microsoft Teams...,HP 6GV21PA Elite Sl

In [220]:
# After you compute df_clean["C_id"] with HDBSCAN:
print("All cluster IDs:", sorted(df_clean["C_id"].unique()))
print(df_clean["C_id"].value_counts().head(10))
#354 product descriptions
#Cluster ID 9 contains 354 rows from your dataframe.
#354 product descriptions as in 354 reords (yaketext 354) which is all clustered as cid 9 

All cluster IDs: [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
C_id
-1     380
 9     354
 45    190
 21    123
 38    113
 5      90
 3      88
 8      85
 27     85
 16     84
Name: count, dtype: int64


In [221]:
def make_evidence(subdf, max_chars=1200, max_items=25):
    """
    Build a compact textual summary for one cluster:
    - use yake_text (already compressed)
    - take up to max_items examples
    - truncate total length to max_chars
    """
    items = []
    for txt in subdf["yake_text"].fillna("").tolist():
        t = txt.strip()
        if not t:
            continue
        items.append("- " + t[:160])   # short snippet per product
        if len(items) >= max_items:
            break

    blob = "\n".join(items)
    return blob[:max_chars]


In [222]:
import subprocess
import hashlib

def run_llm(prompt, model="llama3.2"):
    cmd = ["ollama", "run", model]
    result = subprocess.run(
        cmd,
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    stderr_text = result.stderr.decode("utf-8", errors="ignore")
    if stderr_text.strip():
        print("⚠ Ollama stderr:\n", stderr_text)
    stdout_text = result.stdout.decode("utf-8", errors="ignore")
    return stdout_text

label_cache = {}

def cached_llm_label(prompt, max_words=4, model="llama3.2"):
    key = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
    if key in label_cache:
        return label_cache[key]

    raw = run_llm(prompt, model=model)
    # take only first few words, strip punctuation
    label = " ".join(raw.strip().split()[:max_words])
    label_cache[key] = label
    return label


In [223]:
import re

def clean_cluster_name(name: str, max_words: int = 3) -> str:
    if not isinstance(name, str):
        return ""

    name = name.strip()
    name = re.sub(r"\b\d+[\.\)]\s*", " ", name)           # remove "1." / "2)"
    name = re.sub(r"[^A-Za-z0-9&/+ ]+", " ", name)        # keep safe chars
    name = re.sub(r"\s+", " ", name).strip()              # collapse spaces

    tokens = name.split(" ")
    tokens = [t for t in tokens if t not in {"&", "/", "-", "|"}]

    if not tokens:
        return ""

    tokens = tokens[:max_words]
    return " ".join(tokens)


C_name_map = {}
unique_cids = sorted(df_clean["C_id"].unique())
print("All cluster IDs:", unique_cids)

for cid in unique_cids:
    subdf = df_clean[df_clean["C_id"] == cid]

    blob = make_evidence(subdf, max_chars=1200, max_items=25)
    if not blob.strip():
        print(f"C_id {cid} → EMPTY, assigning 'Other Category'")
        C_name_map[cid] = "Other Category"
        continue

    prompt = f"""
[SYSTEM]
Your task is to assign a clean, generic product category name.
Max 3 words. No brands. No specs. No punctuation. Only the category type.
[USER]
Here are example products from one cluster:

{blob}

Respond with ONLY the final 3-word-maximum category name. No explanations.
"""

    raw_label = cached_llm_label(prompt, max_words=6)
    cleaned_label = clean_cluster_name(raw_label, max_words=3)

    if not cleaned_label:
        cleaned_label = "Other Category"

    C_name_map[cid] = cleaned_label
    print(f"C_id {cid} :  '{cleaned_label}'")




All cluster IDs: [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ ⠴ ⠦ ⠦ ⠇ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠦ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠋ ⠋ ⠹ ⠸ ⠸ ⠼ ⠴ ⠦ ⠧ ⠏ ⠋ ⠋ ⠹ ⠹ ⠼ ⠴ ⠴ ⠧ ⠇ ⠏ ⠋ ⠋ ⠹ ⠸ ⠸ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠙ ⠸ ⠸ ⠴ ⠦ ⠦ ⠇ ⠇ ⠋ ⠙ ⠹ ⠸ ⠸ ⠴ ⠦ ⠦ ⠇ 
C_id -1 :  'Computer Hardware and'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
C_id 0 :  'Laptop Screen Replacement'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
C_id 1 :  'Display Tablet Computers'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ 
C_id 2 :  'Laptop Screen Replacement'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠼ 
C_id 3 :  'Satellite LCD Screen'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠼ 
C_id 4 :  'Rechargeable Laptop Batteries'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠼ 
C_id 5 :  'Laptop Screen Replacement'
⚠ Ollama stderr:
 ⠋ ⠹ ⠸ ⠼ 
C_id 6 :  'Computing and Electronics'
⚠ Oll

In [226]:
# Attach ONLY the clean name
df_clean["C_name"] = df_clean["C_id"].map(C_name_map)

# (Optional) check
df_clean[["C_id", "C", "C_name"]].head(20)

,C_id,C,C_name
1072689,66,PCs/Workstations,Personal Computing Devices
906402,49,Notebook Parts & Accessories,Computer Keyboards
411281,48,Fibre Optic Cables,Fiber Optic Cables
1047582,77,PCs/Workstations,Computer Desktop Systems
904910,49,Notebook Parts & Accessories,Computer Keyboards
157385,15,Software Licenses/Upgrades,Computer Server Support
934548,62,Notebook Parts & Accessories,Computer Keyboards
876762,38,Notebook Parts & Accessories,Computer Hardware Parts
1041385,9,Printing Supplies,Office Supplies
273638,45,Computer Monitors,Computer Monitors


In [227]:
df_clean.head(2)

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,meta_len,yake_k,yake_keywords,yake_text,C_id,C_name
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,3657,30,"[ASUS, Intel, Windows, Core, Graphics, Cache, ...",ASUS Intel Windows Core Graphics Cache RAM Eth...,66,Personal Computing Devices
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...,299,10,"[Keyboard, Belgium, backlight, cable, includes...",Keyboard Belgium backlight cable includes comp...,49,Computer Keyboards


In [228]:
# C → B: cluster C-centroids into B_id
from sklearn.cluster import AgglomerativeClustering
import numpy as np

# 1. Compute centroids for each C_id
valid_cids = sorted(df_clean["C_id"].unique())

C_centroids = []
C_labels = []

for cid in valid_cids:
    idx = df_clean["C_id"] == cid
    C_centroids.append(reduced[idx].mean(axis=0))   # centroid in UMAP space
    C_labels.append(cid)

C_centroids = np.vstack(C_centroids)

# 2. Agglomerative clustering on C-centroids → B-level groups
agg_B = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.0,   # you can tune 0.8–1.2 later
    linkage="ward"
)

B_ids = agg_B.fit_predict(C_centroids)

# 3. Map C_id → B_id and attach to df_clean
C_to_B = dict(zip(C_labels, B_ids))
df_clean["B_id"] = df_clean["C_id"].map(lambda c: C_to_B.get(c, -1))

print(df_clean[["C_id", "C_name", "B_id"]].head())
df_clean["B_id"].value_counts().head()


         C_id                      C_name  B_id
1072689    66  Personal Computing Devices    28
906402     49          Computer Keyboards    10
411281     48          Fiber Optic Cables     4
1047582    77    Computer Desktop Systems     7
904910     49          Computer Keyboards    10


B_id
43    380
41    354
14    232
8     216
6     155
Name: count, dtype: int64

In [229]:
import re

# if not already defined, reuse this cleaner:
def clean_cluster_name(name: str, max_words: int = 3) -> str:
    if not isinstance(name, str):
        return ""

    name = name.strip()
    name = re.sub(r"\b\d+[\.\)]\s*", " ", name)          # remove "1." / "2)"
    name = re.sub(r"[^A-Za-z0-9&/+ ]+", " ", name)       # keep safe chars
    name = re.sub(r"\s+", " ", name).strip()

    tokens = [t for t in name.split(" ") if t not in {"&", "/", "-", "|"}]
    if not tokens:
        return ""

    tokens = tokens[:max_words]
    return " ".join(tokens)


def name_B_clusters(df):
    B_name_map = {}

    for bid in sorted(df["B_id"].unique()):
        if bid == -1:
            continue

        c_names = (
            df.loc[df["B_id"] == bid, "C_name"]
              .dropna()
              .unique()
              .tolist()
        )
        if not c_names:
            B_name_map[bid] = "Other Category"
            continue

        sample_names = ", ".join(c_names[:15])

        prompt = f"""
[SYSTEM]
You are an expert in product taxonomy.
You see a list of C-level category names and must assign
a concise B-level group name.

STRICT RULES:
- Maximum 3 words.
- No brand names.
- Generic, in English.
- Single line with ONLY the category name.

[USER]
Here are C-level subcategories:

{sample_names}

Respond with ONLY the 3-word-maximum B-level category name.
"""

        raw_label = cached_llm_label(prompt, max_words=6)
        cleaned_label = clean_cluster_name(raw_label, max_words=3)
        if not cleaned_label:
            cleaned_label = "Other Category"

        B_name_map[bid] = cleaned_label
        print(f"B_id {bid} : '{cleaned_label}'")

    return B_name_map


B_name_map = name_B_clusters(df_clean)
df_clean["B_name"] = df_clean["B_id"].map(B_name_map)

df_clean[["C_id", "C_name", "B_id", "B_name"]].head(15)


⚠ Ollama stderr:
 ⠋ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ 
B_id 0 : 'Information Security'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ 
B_id 1 : 'Phone Case Accessories'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ 
B_id 2 : 'Desktop Computing Products'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ 
B_id 3 : 'IT Help Desk'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
B_id 4 : 'Computer Network Equipment'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠼ 
B_id 5 : 'Computer Display Products'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
B_id 6 : 'Computer and Mobile'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
B_id 7 : 'Desktop Computers'
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠼ 
B_id 8 : 'Display Technology Products'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
B_id 9 : 'Electronics Computer'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠼ 
B_id 10 : 'Input and Output'
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ ⠸ 
B_id 11 : 'Consumer Electronics'
⚠ Ollama stderr:
 ⠋ ⠹ ⠸ ⠼ 
B_id 12 : 'Mobile Computing Hardware'
⚠ Ollama stderr:
 ⠋ ⠙ ⠸ ⠼ 
B_id 13 : 'Cybersecurity Product Software'
⚠ Ollama stderr:
 ⠙ ⠙ ⠹ ⠸ 
B_id 14 : 'Storage Devices and'
B_id 15 : 'Input and Output'
B_id 16 : 'Input and Output'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ 

,C_id,C_name,B_id,B_name
1072689,66,Personal Computing Devices,28,Computer Hardware Components
906402,49,Computer Keyboards,10,Input and Output
411281,48,Fiber Optic Cables,4,Computer Network Equipment
1047582,77,Computer Desktop Systems,7,Desktop Computers
904910,49,Computer Keyboards,10,Input and Output
157385,15,Computer Server Support,22,Server Technical Support
934548,62,Computer Keyboards,15,Input and Output
876762,38,Computer Hardware Parts,17,Peripherals and Accessories
1041385,9,Office Supplies,41,Stationery and Office
273638,45,Computer Monitors,8,Display Technology Products


In [230]:
df_clean.head(2)

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,meta_len,yake_k,yake_keywords,yake_text,C_id,C_name,B_id,B_name
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,ASUS K31CD-IT049T K31CD-IT049T Intel Core i7-6...,3657,30,"[ASUS, Intel, Windows, Core, Graphics, Cache, ...",ASUS Intel Windows Core Graphics Cache RAM Eth...,66,Personal Computing Devices,28,Computer Hardware Components
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,HP 686915-A41 686915-A41 Keyboard in midnight ...,HP 686915-A41 686915-A41 Keyboard in midnight ...,299,10,"[Keyboard, Belgium, backlight, cable, includes...",Keyboard Belgium backlight cable includes comp...,49,Computer Keyboards,10,Input and Output


In [231]:
from sklearn.cluster import AgglomerativeClustering
import numpy as np

valid_bids = [b for b in sorted(df_clean["B_id"].unique()) if b != -1]

B_centroids = []
B_labels = []

for bid in valid_bids:
    idx = (df_clean["B_id"] == bid)
    B_centroids.append(reduced[idx].mean(axis=0))   # UMAP centroid
    B_labels.append(bid)

B_centroids = np.vstack(B_centroids)

agg_A = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.2,   # you can tune 1.0–1.5 later
    linkage="ward"
)

A_ids = agg_A.fit_predict(B_centroids)

B_to_A = dict(zip(B_labels, A_ids))
df_clean["A_id"] = df_clean["B_id"].map(lambda b: B_to_A.get(b, -1))

df_clean[["A_id", "B_id", "B_name"]].head()
df_clean["A_id"].value_counts().head()


A_id
20    380
19    354
3     275
5     265
25    216
Name: count, dtype: int64

In [232]:
def name_A_clusters(df):
    A_name_map = {}

    for aid in sorted(df["A_id"].unique()):
        if aid == -1:
            continue

        b_names = (
            df.loc[df["A_id"] == aid, "B_name"]
              .dropna()
              .unique()
              .tolist()
        )
        if not b_names:
            A_name_map[aid] = "Other Group"
            continue

        sample_names = ", ".join(b_names[:15])

        prompt = f"""
[SYSTEM]
You are an expert in product taxonomy.
You see a list of B-level category names and must assign
a concise A-level (top-level) group name.

STRICT RULES:
- Maximum 3 words.
- No brand names.
- Generic, in English.
- Single line with ONLY the category name.

[USER]
Here are B-level categories:

{sample_names}

Respond with ONLY the 3-word-maximum A-level category name.
"""

        raw_label = cached_llm_label(prompt, max_words=6)
        cleaned_label = clean_cluster_name(raw_label, max_words=3)
        if not cleaned_label:
            cleaned_label = "Other Group"

        A_name_map[aid] = cleaned_label
        print(f"A_id {aid} : '{cleaned_label}'")

    return A_name_map


A_name_map = name_A_clusters(df_clean)
df_clean["A_name"] = df_clean["A_id"].map(A_name_map)

df_clean[["A_id", "A_name", "B_id", "B_name", "C_id", "C_name"]].head(20)


⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ ⠴ ⠦ 
A_id 0 : 'Mobile Computing Hardware'
⚠ Ollama stderr:
 ⠙ ⠙ ⠹ 
A_id 1 : 'Computer and Technology'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ 
A_id 2 : 'Computer Peripherals'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠼ 
A_id 3 : 'Storage and Organization'
⚠ Ollama stderr:
 ⠙ ⠙ ⠸ 
A_id 4 : 'Display Repair Services'
⚠ Ollama stderr:
 ⠙ ⠹ ⠹ ⠼ 
A_id 5 : 'Computer Hardware Products'
⚠ Ollama stderr:
 ⠙ ⠹ ⠸ ⠸ 
A_id 6 : 'Computing Hardware'
⚠ Ollama stderr:
 ⠙ ⠙ ⠹ 
A_id 7 : 'Camera Equipment'
A_id 8 : 'Display Repair Services'
A_id 9 : 'Display Repair Services'
⚠ Ollama stderr:
 ⠋ ⠹ ⠹ ⠼ 
A_id 10 : 'Consumer Electronics'
⚠ Ollama stderr:
 ⠙ ⠙ 
A_id 11 : 'Computing and Gadgets'
⚠ Ollama stderr:
 ⠙ ⠙ 
A_id 12 : 'Ear Monitoring Systems'
⚠ Ollama stderr:
 ⠙ ⠹ ⠹ 
A_id 13 : 'Security Software Solutions'
⚠ Ollama stderr:
 ⠙ ⠹ ⠹ 
A_id 14 : 'Personal Mobile Devices'
⚠ Ollama stderr:
 ⠙ ⠙ ⠹ ⠼ 
A_id 15 : 'Computer Components'
⚠ Ollama stderr:
 ⠋ ⠹ ⠸ ⠸ 
A_id 16 : 'Computer Peripherals'
A_id 17 : 'Display Repair

,A_id,A_name,B_id,B_name,C_id,C_name
1072689,24,Computer Peripherals,28,Computer Hardware Components,66,Personal Computing Devices
906402,39,Data Exchange Standards,10,Input and Output,49,Computer Keyboards
411281,34,Network Infrastructure,4,Computer Network Equipment,48,Fiber Optic Cables
1047582,30,Personal Computing Devices,7,Desktop Computers,77,Computer Desktop Systems
904910,39,Data Exchange Standards,10,Input and Output,49,Computer Keyboards
157385,38,Server Hardware Support,22,Server Technical Support,15,Computer Server Support
934548,33,Data Exchange Standards,15,Input and Output,62,Computer Keyboards
876762,16,Computer Peripherals,17,Peripherals and Accessories,38,Computer Hardware Parts
1041385,19,Office Supplies,41,Stationery and Office,9,Office Supplies
273638,25,Electronics Equipment,8,Display Technology Products,45,Computer Monitors


In [233]:
hierarchy = (
    df_clean
      .dropna(subset=["A_name", "B_name", "C_name"])
      .groupby(["A_name", "B_name", "C_name"])
      .size()
      .reset_index(name="num_products")
      .sort_values(["A_name", "B_name", "num_products"],
                   ascending=[True, True, False])
)

hierarchy.head(30)   # inspect


,A_name,B_name,C_name,num_products
2,Accessories for Phones,Phone Case Accessories,Tablet Computer Accessories,77
0,Accessories for Phones,Phone Case Accessories,Computer Accessories Cases,35
1,Accessories for Phones,Phone Case Accessories,Screen Phone Protectors,30
3,Battery Charging Solutions,Portable Power Supplies,Rechargeable Laptop Batteries,38
4,Camera Equipment,Digital Camera Accessories,Camera Equipment,36
5,Computer Accessories,Peripherals and Devices,Computer Components,83
6,Computer Components,Microprocessor Units,Computer Processors,63
7,Computer Hardware,Gaming Computer Systems,Computer Gaming Hardware,73
8,Computer Hardware Components,Computer Power Supplies,Power Supplies,43
10,Computer Hardware Products,Desktop Computing Products,Laptop Computer Accessories,50


In [234]:
from anytree import Node, RenderTree

# 1) Take unique A–B–C rows from your 'hierarchy' df
unique_paths = (
    hierarchy[["A_name", "B_name", "C_name", "num_products"]]
    .dropna()
    .drop_duplicates()
)

# 2) Create root (rename if you like)
root = Node("Computers & Electronics")

# 3) Cache to avoid recreating nodes
node_cache = {}   # keys: ("A", A_name), ("B", A_name, B_name), ("C", A_name, B_name, C_name)

for _, row in unique_paths.iterrows():
    a = row["A_name"]
    b = row["B_name"]
    c = row["C_name"]
    n = row["num_products"]

    # ---- A level ----
    a_key = ("A", a)
    if a_key not in node_cache:
        node_cache[a_key] = Node(a, parent=root)
    a_node = node_cache[a_key]

    # ---- B level ----
    b_key = ("B", a, b)
    if b_key not in node_cache:
        node_cache[b_key] = Node(b, parent=a_node)
    b_node = node_cache[b_key]

    # ---- C level (with counts) ----
    c_key = ("C", a, b, c)
    if c_key not in node_cache:
        label_c = f"{c} ({n} products)"
        node_cache[c_key] = Node(label_c, parent=b_node)

# 4) Pretty-print tree
for pre, _, node in RenderTree(root):
    print(f"{pre}{node.name}")


Computers & Electronics
├── Accessories for Phones
│   └── Phone Case Accessories
│       ├── Tablet Computer Accessories (77 products)
│       ├── Computer Accessories Cases (35 products)
│       └── Screen Phone Protectors (30 products)
├── Battery Charging Solutions
│   └── Portable Power Supplies
│       └── Rechargeable Laptop Batteries (38 products)
├── Camera Equipment
│   └── Digital Camera Accessories
│       └── Camera Equipment (36 products)
├── Computer Accessories
│   └── Peripherals and Devices
│       └── Computer Components (83 products)
├── Computer Components
│   └── Microprocessor Units
│       └── Computer Processors (63 products)
├── Computer Hardware
│   └── Gaming Computer Systems
│       └── Computer Gaming Hardware (73 products)
├── Computer Hardware Components
│   └── Computer Power Supplies
│       └── Power Supplies (43 products)
├── Computer Hardware Products
│   ├── Desktop Computing Products
│   │   ├── Laptop Computer Accessories (50 products)
│   │   ├─

In [235]:
from anytree import Node, RenderTree

# 1) Take unique A–B–C rows
unique_paths = (
    hierarchy[["A_name", "B_name", "C_name", "num_products"]]
    .dropna()
    .drop_duplicates()
)

# 2) We create multiple A-nodes directly (no global root)
roots = {}     # {A_name : Node}

node_cache = {}  # prevent duplicates

for _, row in unique_paths.iterrows():
    a = row["A_name"]
    b = row["B_name"]
    c = row["C_name"]
    n = row["num_products"]

    # ---- A level (each A is its own root) ----
    if a not in roots:
        roots[a] = Node(a)
    a_node = roots[a]

    # ---- B level ----
    b_key = ("B", a, b)
    if b_key not in node_cache:
        node_cache[b_key] = Node(b, parent=a_node)
    b_node = node_cache[b_key]

    # ---- C level ----
    c_key = ("C", a, b, c)
    if c_key not in node_cache:
        label_c = f"{c} ({n} products)"
        node_cache[c_key] = Node(label_c, parent=b_node)

# 3) Print each A-tree separately
for a, root_node in roots.items():
    print(f"\n=== {a} ===")
    for pre, _, node in RenderTree(root_node):
        print(f"{pre}{node.name}")



=== Accessories for Phones ===
Accessories for Phones
└── Phone Case Accessories
    ├── Tablet Computer Accessories (77 products)
    ├── Computer Accessories Cases (35 products)
    └── Screen Phone Protectors (30 products)

=== Battery Charging Solutions ===
Battery Charging Solutions
└── Portable Power Supplies
    └── Rechargeable Laptop Batteries (38 products)

=== Camera Equipment ===
Camera Equipment
└── Digital Camera Accessories
    └── Camera Equipment (36 products)

=== Computer Accessories ===
Computer Accessories
└── Peripherals and Devices
    └── Computer Components (83 products)

=== Computer Components ===
Computer Components
└── Microprocessor Units
    └── Computer Processors (63 products)

=== Computer Hardware ===
Computer Hardware
└── Gaming Computer Systems
    └── Computer Gaming Hardware (73 products)

=== Computer Hardware Components ===
Computer Hardware Components
└── Computer Power Supplies
    └── Power Supplies (43 products)

=== Computer Hardware Produ

In [236]:
# =============== PRETTY ASCII TREE PRINTER ===============

def print_ascii_tree(hierarchy_df):
    """
    hierarchy_df must have columns: A_name, B_name, C_name, num_products
    """
    
    # Step 1: group hierarchy by A
    A_groups = hierarchy_df.groupby("A_name")

    for A_name, dfA in A_groups:
        print(f"\n{A_name}")

        # Create a prefix for A-level children
        A_prefix = ""

        # Step 2: group B under each A
        B_groups = dfA.groupby("B_name")

        B_names = list(B_groups.groups.keys())
        last_B = B_names[-1]

        for B_name, dfB in B_groups:
            is_last_B = (B_name == last_B)
            B_connector = "└──" if is_last_B else "├──"
            print(f"{A_prefix}{B_connector} {B_name}")

            # prefix for C under this B
            C_prefix = "    " if is_last_B else "│   "

            # Step 3: C-level
            C_rows = dfB[["C_name", "num_products"]].values.tolist()
            last_C = C_rows[-1]

            for C_name, num_products in C_rows:
                is_last_C = (C_name == last_C[0])
                C_connector = "└──" if is_last_C else "├──"
                print(f"{A_prefix}{C_prefix}{C_connector} {C_name} ({num_products})")


In [237]:
print_ascii_tree(hierarchy)



Accessories for Phones
└── Phone Case Accessories
    ├── Tablet Computer Accessories (77)
    ├── Computer Accessories Cases (35)
    └── Screen Phone Protectors (30)

Battery Charging Solutions
└── Portable Power Supplies
    └── Rechargeable Laptop Batteries (38)

Camera Equipment
└── Digital Camera Accessories
    └── Camera Equipment (36)

Computer Accessories
└── Peripherals and Devices
    └── Computer Components (83)

Computer Components
└── Microprocessor Units
    └── Computer Processors (63)

Computer Hardware
└── Gaming Computer Systems
    └── Computer Gaming Hardware (73)

Computer Hardware Components
└── Computer Power Supplies
    └── Power Supplies (43)

Computer Hardware Products
├── Desktop Computing Products
│   ├── Laptop Computer Accessories (50)
│   ├── Computer Hardware (40)
│   └── Laptops and Tablets (30)
└── Electronics Computer
    ├── Computer Hardware Components (46)
    ├── Electronics Computer Devices (43)
    ├── Computer Hardware Products (29)
    └──

In [238]:
#Check cosine similarity between path_3 and predicted taxonomy( how similar they are)

In [240]:
# Normalize all labels (gold + predicted)
import re

def normalize_text(s):
    if not isinstance(s, str):
        return ""
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# Gold
df_clean["gold_A_norm"] = df_clean["A"].apply(normalize_text)
df_clean["gold_B_norm"] = df_clean["B"].apply(normalize_text)
df_clean["gold_C_norm"] = df_clean["C"].apply(normalize_text)

# Predicted
df_clean["pred_A_norm"] = df_clean["A_name"].apply(normalize_text)
df_clean["pred_B_norm"] = df_clean["B_name"].apply(normalize_text)
df_clean["pred_C_norm"] = df_clean["C_name"].apply(normalize_text)

# Full path (gold vs discovered)
df_clean["gold_path_norm"] = df_clean["path_3"].apply(normalize_text)
df_clean["pred_path_norm"] = (
    (df_clean["A_name"].fillna("") + " > " +
     df_clean["B_name"].fillna("") + " > " +
     df_clean["C_name"].fillna(""))
    .apply(normalize_text)
)


In [241]:
#Load MiniLM and define helper for cosine
from sentence_transformers import SentenceTransformer
import numpy as np

st = SentenceTransformer("all-MiniLM-L6-v2")

def mean_cosine_sim(texts_gold, texts_pred):
    gold_emb = st.encode(texts_gold, normalize_embeddings=True)
    pred_emb = st.encode(texts_pred, normalize_embeddings=True)
    cos = np.sum(gold_emb * pred_emb, axis=1)   # dot product = cosine
    return float(cos.mean())


In [242]:
# Cosine similarity for A, B, C, full path
cos_A = mean_cosine_sim(
    df_clean["gold_A_norm"].tolist(),
    df_clean["pred_A_norm"].tolist()
)

cos_B = mean_cosine_sim(
    df_clean["gold_B_norm"].tolist(),
    df_clean["pred_B_norm"].tolist()
)

cos_C = mean_cosine_sim(
    df_clean["gold_C_norm"].tolist(),
    df_clean["pred_C_norm"].tolist()
)

cos_path = mean_cosine_sim(
    df_clean["gold_path_norm"].tolist(),
    df_clean["pred_path_norm"].tolist()
)

print("A-level cosine similarity:", cos_A)
print("B-level cosine similarity:", cos_B)
print("C-level cosine similarity:", cos_C)
print("Full-path cosine similarity:", cos_path)


A-level cosine similarity: 0.5126641392707825
B-level cosine similarity: 0.48224878311157227
C-level cosine similarity: 0.47320079803466797
Full-path cosine similarity: 0.5201599597930908


In [243]:
#purity / NMI are to check how well your HDBSCAN C-clusters line up with the original C-level labels (ignoring the LLM names).
# Think:
# C_id = discovered cluster from HDBSCAN
# C = gold C-level from original taxonomy

from sklearn.metrics import normalized_mutual_info_score
import numpy as np

# Keep only rows that have both a cluster and a gold C-level
df_eval = df_clean.dropna(subset=["C_id", "C"]).copy()

# If any -1 noise still exists, drop it for evaluation
df_eval = df_eval[df_eval["C_id"] != -1].copy()

y_clusters = df_eval["C_id"].astype(int).to_numpy()
y_gold_C  = df_eval["C"].astype(str).to_numpy()


In [244]:
def cluster_purity(y_true, y_pred):
    """
    y_true: gold labels (here: C)
    y_pred: discovered clusters (here: C_id)
    """
    # unique clusters
    clusters = np.unique(y_pred)
    N = len(y_pred)
    correct = 0

    for c in clusters:
        idx = (y_pred == c)
        if not np.any(idx):
            continue
        # gold labels inside this cluster
        labels, counts = np.unique(y_true[idx], return_counts=True)
        correct += counts.max()   # majority label count

    return correct / N

purity_C = cluster_purity(y_gold_C, y_clusters)
print("C-level cluster purity:", purity_C)
# Cluster Purity = “How clean is each cluster inside?”
# Your purity: 0.826 (~82.6%)
# Purity checks within each discovered HDBSCAN cluster:
# “If I look inside cluster X, do most products belong to the same gold C-level category?”
# For example:
# Cluster 9 has 354 products
# Suppose 290 of them are “Laptop Screens” (majority)
# Purity contribution = 290 / 354

C-level cluster purity: 0.8255872063968016


In [245]:
nmi_C = normalized_mutual_info_score(y_gold_C, y_clusters)
print("C-level NMI (gold C vs C_id):", nmi_C)
#It measures how close the entire clustering assignment is to the gold categories.
# @“How well does the entire clustering structure match the gold taxonomy

C-level NMI (gold C vs C_id): 0.6672069526713204


In [246]:
#BERTSCORE

In [247]:
# Normalize gold labels
df_clean["gold_A"] = df_clean["A"].fillna("").astype(str).str.strip()
df_clean["gold_B"] = df_clean["B"].fillna("").astype(str).str.strip()
df_clean["gold_C"] = df_clean["C"].fillna("").astype(str).str.strip()

# Normalize predicted labels
df_clean["pred_A"] = df_clean["A_name"].fillna("").astype(str).str.strip()
df_clean["pred_B"] = df_clean["B_name"].fillna("").astype(str).str.strip()
df_clean["pred_C"] = df_clean["C_name"].fillna("").astype(str).str.strip()

# Build gold path and predicted path
df_clean["gold_path"] = (
    df_clean["gold_A"] + " > " +
    df_clean["gold_B"] + " > " +
    df_clean["gold_C"]
)

df_clean["pred_path"] = (
    df_clean["pred_A"] + " > " +
    df_clean["pred_B"] + " > " +
    df_clean["pred_C"]
)


In [248]:
from bert_score import score as bertscore_score

def compute_bertscore(refs, cands, model_type="roberta-large"):
    # Replace any empty strings with "none"
    refs = ["none" if not r.strip() else r for r in refs]
    cands = ["none" if not c.strip() else c for c in cands]

    P, R, F1 = bertscore_score(
        cands,  # candidates
        refs,   # references (gold)
        model_type="bert-base-uncased",
        lang="en",
        rescale_with_baseline=False
    )
    return float(P.mean()), float(R.mean()), float(F1.mean())


In [249]:
gold_A = df_clean["gold_A"].tolist()
pred_A = df_clean["pred_A"].tolist()

gold_B = df_clean["gold_B"].tolist()
pred_B = df_clean["pred_B"].tolist()

gold_C = df_clean["gold_C"].tolist()
pred_C = df_clean["pred_C"].tolist()

gold_path = df_clean["gold_path"].tolist()
pred_path = df_clean["pred_path"].tolist()


In [250]:
print(" A-level BERTScore:")
print(compute_bertscore(gold_A, pred_A))

print("\n B-level BERTScore:")
print(compute_bertscore(gold_B, pred_B))

print("\n C-level BERTScore:")
print(compute_bertscore(gold_C, pred_C))

print("\n Full-path BERTScore:")
print(compute_bertscore(gold_path, pred_path))


 A-level BERTScore:
(0.5946646332740784, 0.5597482919692993, 0.5757115483283997)

 B-level BERTScore:
(0.5408266186714172, 0.6198215484619141, 0.5741963386535645)

 C-level BERTScore:
(0.598725438117981, 0.5848424434661865, 0.589326798915863)

 Full-path BERTScore:
(0.6769204139709473, 0.6764034032821655, 0.6758546829223633)


In [251]:
#how similar is our generated tree to the test data

In [252]:
df_test = pd.read_json("D:/git/Taxonomy_Buidling_Textual_Corpora/data/icecat_data_test.json")

# Work only with first 5000 rows from the beginning
df_test = df_test.iloc[:2000].copy()

print("Raw sample size:", len(df_test))


Raw sample size: 2000


In [253]:
df_test.head(2)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
500081,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...
741063,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,EN,13,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks


In [254]:
# From TRAIN (df_clean)
import numpy as np

# centroids in UMAP space, one vector per C_id
valid_cids = sorted(df_clean["C_id"].unique())
C_centroids = []
C_labels = []

for cid in valid_cids:
    idx = df_clean["C_id"] == cid
    C_centroids.append(df_clean.loc[idx, "umap_1":"umap_2"].to_numpy().mean(axis=0) 
                       if "umap_1" in df_clean.columns
                       else reduced[idx].mean(axis=0))
    C_labels.append(cid)

C_centroids = np.vstack(C_centroids)  # shape (#C_clusters, 2) if UMAP(2D)
C_labels = np.array(C_labels)

# Build mappings for B, A based on TRAIN
C_to_B = dict(zip(df_clean["C_id"], df_clean["B_id"]))
B_to_A = dict(zip(df_clean["B_id"], df_clean["A_id"]))

B_name_map = dict(zip(df_clean["B_id"], df_clean["B_name"]))
A_name_map = dict(zip(df_clean["A_id"], df_clean["A_name"]))


In [255]:
df_clean.head(2)

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,...,gold_path_norm,pred_path_norm,gold_A,gold_B,gold_C,pred_A,pred_B,pred_C,gold_path,pred_path
1072689,ASUS,K31CD-IT049T,K31CD-IT049T,"Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,...,computers electronics computers pcs workstations,computer peripherals computer hardware compone...,Computers & Electronics,Computers,PCs/Workstations,Computer Peripherals,Computer Hardware Components,Personal Computing Devices,Computers & Electronics > Computers > PCs/Work...,Computer Peripherals > Computer Hardware Compo...
906402,HP,686915-A41,686915-A41,Keyboard in midnight black finish with backlig...,HP 686915-A41. Type: Keyboard. Keyboard langua...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,...,computers electronics computers notebook parts...,data exchange standards input and output compu...,Computers & Electronics,Computers,Notebook Parts & Accessories,Data Exchange Standards,Input and Output,Computer Keyboards,Computers & Electronics > Computers > Notebook...,Data Exchange Standards > Input and Output > C...


In [256]:
def split_path(path):
    if not isinstance(path, str):
        return []
    parts = [p.strip() for p in path.split(">") if p.strip()]
    return parts

df_test["num_levels"] = df_test["pathlist_names"].apply(lambda x: len(split_path(x)))

print("Counts BEFORE cleaning:")
print(df_test["num_levels"].value_counts().sort_index())

Counts BEFORE cleaning:
num_levels
3    1198
4     802
Name: count, dtype: int64


In [257]:
import pandas as pd

def make_3_and_4(path):
    parts = split_path(path)

    # default
    path_3 = None
    level_4 = None

    if len(parts) >= 3:
        path_3 = " > ".join(parts[:3])   # A > B > C
    if len(parts) >= 4:
        level_4 = parts[3]               # D (4th level)

    return pd.Series({"path_3": path_3, "level_4": level_4})

df_test[["path_3", "level_4"]] = df_test["pathlist_names"].apply(make_3_and_4)

print(df_test[["pathlist_names", "path_3", "level_4"]].head(10))


                                            pathlist_names  \
500081   Computers & Electronics>Warranty & Support>War...   
741063         Computers & Electronics>Computers>Notebooks   
1091454  Computers & Electronics>Computers>PCs/Workstat...   
522928   Computers & Electronics>Computer Cables>Networ...   
479478   Computers & Electronics>Warranty & Support>War...   
449642   Computers & Electronics>Data Storage>Data Stor...   
1024163  Computers & Electronics>Consumer Audio & Video...   
31904    Computers & Electronics>Printers & Scanners>Pr...   
360523   Computers & Electronics>Telecom & Navigation>M...   
675402         Computers & Electronics>Computers>Notebooks   

                                                    path_3  \
500081   Computers & Electronics > Warranty & Support >...   
741063     Computers & Electronics > Computers > Notebooks   
1091454  Computers & Electronics > Computers > PCs/Work...   
522928   Computers & Electronics > Computer Cables > Ne...   
479478 

In [258]:
print("Rows:", len(df))
print("Unique num_levels (original):")
print(df_test["num_levels"].value_counts().sort_index())

print("\nCheck how many have a 4th level stored:")
print(df_test["level_4"].notna().sum(), "rows with level_4")


Rows: 4382
Unique num_levels (original):
num_levels
3    1198
4     802
Name: count, dtype: int64

Check how many have a 4th level stored:
802 rows with level_4


In [260]:
# Extract A, B, C from path_3
df_test[["A", "B", "C"]] = (
    df_test["path_3"]
    .str.split(">", expand=True)
    .apply(lambda col: col.str.strip())
)


In [261]:
df_test.head(2)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names,num_levels,path_3,level_4,A,B,C
500081,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...,3,Computers & Electronics > Warranty & Support >...,None,Computers & Electronics,Warranty & Support,Warranty & Support Extensions
741063,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,"[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks,3,Computers & Electronics > Computers > Notebooks,None,Computers & Electronics,Computers,Notebooks


In [262]:
#columns to keep
cols_needed = [
    "Brand",
    "BrandPartCode",
    "ProductName",
    "Description.LongProductName",
    "SummaryDescription.LongSummaryDescription",
    "Description.LongDesc",
    "A", "B", "C",
    "path_3"   # <-- Needed for evaluation
]


In [263]:
df_test = df_test[cols_needed].copy()


In [264]:
# Build raw metadata text

def build_metadata(row):
    parts = []

    if pd.notna(row["Brand"]):
        parts.append(row["Brand"])

    if pd.notna(row["BrandPartCode"]):
        parts.append(row["BrandPartCode"])
        
    if pd.notna(row["ProductName"]):
        parts.append(row["ProductName"])

    if pd.notna(row["Description.LongProductName"]):
        parts.append(row["Description.LongProductName"])

    if pd.notna(row["SummaryDescription.LongSummaryDescription"]):
        parts.append(row["SummaryDescription.LongSummaryDescription"])

    if pd.notna(row["Description.LongDesc"]):
        parts.append(row["Description.LongDesc"])

    return " ".join(parts)

df_test["metadata_text"] = df_test.apply(build_metadata, axis=1)


In [265]:
df_test

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text
500081,Fujitsu,FSP:G-SW3Z560PRE0S,FSP:G-SW3Z560PRE0S,"SP 3y TS Sub & Upgr, 9x5, 4h Rm Rt f/ CS200c A...",Fujitsu FSP:G-SW3Z560PRE0S. Number of years: 3...,In times of growing complexity and decreasing ...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,Fujitsu FSP:G-SW3Z560PRE0S FSP:G-SW3Z560PRE0S ...
741063,HP,5KM83PA,13-ap0023tu,"Intel® Core™ i7-8565U (1.8 GHz), 16GB DDR4-SDR...",HP Spectre x360 13-ap0023tu. Product type: Hyb...,<b>Revolutionary battery life on a convertible...,Computers & Electronics,Computers,Notebooks,Computers & Electronics > Computers > Notebooks,HP 5KM83PA 13-ap0023tu Intel® Core™ i7-8565U (...
1091454,HP,4NG33EA,880-156nf,None,HP OMEN 880-156nf. Processor frequency: 3.2 GH...,None,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,HP 4NG33EA 880-156nf HP OMEN 880-156nf. Proces...
522928,C2G,83061,1m Cat5e Non-Booted Unshielded (UTP) Network P...,Cat5E Assembled UTP Patch Cable Green 1m,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,Perfect for your home office or a large instal...,Computers & Electronics,Computer Cables,Networking Cables,Computers & Electronics > Computer Cables > Ne...,C2G 83061 1m Cat5e Non-Booted Unshielded (UTP)...
479478,Lenovo,5PS0A14091,5PS0A14091,3YR Onsite + Keep Your Drive,"Lenovo 5PS0A14091. Number of years: 3 year(s),...",Lenovo offers a comprehensive portfolio of val...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,Lenovo 5PS0A14091 5PS0A14091 3YR Onsite + Keep...
...,...,...,...,...,...,...,...,...,...,...,...
859244,QNAP,TS-1635AX-4G/48TB-IWPRO,TS-1635AX-4G/48TB-IWPRO 16 Bay NAS,Marvell ARMADA 8040 ARMv8 Cortex-A72 quad-core...,QNAP TS-1635AX-4G/48TB-IWPRO 16 Bay NAS. Insta...,The TS-1635AX is powered by a Marvell® ARMADA®...,Computers & Electronics,Data Storage,Data Storage Devices,Computers & Electronics > Data Storage > Data ...,QNAP TS-1635AX-4G/48TB-IWPRO TS-1635AX-4G/48TB...
7400,Eaton,PW104BA0UA67,Powerware ePDU Basic 0U 16A C13 x 16,Powerware ePDU Basic 0U 16A C13 x 16,Eaton Powerware ePDU Basic 0U 16A C13 x 16. Ra...,Defining a ePDU<br>\r\n-Enclosure based Power ...,Computers & Electronics,Batteries & Power Supplies,Power Distribution Units (PDUs),Computers & Electronics > Batteries & Power Su...,Eaton PW104BA0UA67 Powerware ePDU Basic 0U 16A...
895533,Lenovo,FRU04Y0249,04Y0249,Keyboard (Portuguese),Lenovo 04Y0249. Type: Keyboard. Keyboard langu...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,Lenovo FRU04Y0249 04Y0249 Keyboard (Portuguese...
1034773,Samsung,CLXR838XK,CLXR838XK,"Toner Cartridge, Black, Laser, 30000 pages",Samsung CLXR838XK. Black toner page yield: 300...,Genuine Samsung black imaging unit for CLX-8380nd,Computers & Electronics,Printers & Scanners,Printing Supplies,Computers & Electronics > Printers & Scanners ...,"Samsung CLXR838XK CLXR838XK Toner Cartridge, B..."


In [266]:
import re

def clean_text(t):
    if not isinstance(t, str):
        return ""

    t = re.sub(r"<[^>]+>", " ", t)          # remove HTML tags
    t = re.sub(r"\s+", " ", t)              # collapse spaces
    t = t.replace("\xa0", " ")              # remove non-breaking
    t = t.strip()

    return t

df_test["metadata_text_clean"] = df_test["metadata_text"].apply(clean_text)


In [267]:
df_test.head(2)

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean
500081,Fujitsu,FSP:G-SW3Z560PRE0S,FSP:G-SW3Z560PRE0S,"SP 3y TS Sub & Upgr, 9x5, 4h Rm Rt f/ CS200c A...",Fujitsu FSP:G-SW3Z560PRE0S. Number of years: 3...,In times of growing complexity and decreasing ...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,Fujitsu FSP:G-SW3Z560PRE0S FSP:G-SW3Z560PRE0S ...,Fujitsu FSP:G-SW3Z560PRE0S FSP:G-SW3Z560PRE0S ...
741063,HP,5KM83PA,13-ap0023tu,"Intel® Core™ i7-8565U (1.8 GHz), 16GB DDR4-SDR...",HP Spectre x360 13-ap0023tu. Product type: Hyb...,<b>Revolutionary battery life on a convertible...,Computers & Electronics,Computers,Notebooks,Computers & Electronics > Computers > Notebooks,HP 5KM83PA 13-ap0023tu Intel® Core™ i7-8565U (...,HP 5KM83PA 13-ap0023tu Intel® Core™ i7-8565U (...


In [268]:
df_test["meta_len"] = df_test["metadata_text_clean"].apply(lambda x: len(str(x)))
df_test["meta_len"].describe()


count     2000.000000
mean      1392.246000
std       1989.771436
min         29.000000
25%        253.500000
50%        679.500000
75%       1832.250000
max      21716.000000
Name: meta_len, dtype: float64

In [269]:
def decide_top_k(meta_len):
    if meta_len < 150:
        return 5
    elif meta_len < 500:
        return 10
    elif meta_len < 1500:
        return 15
    elif meta_len < 3000:
        return 20
    elif meta_len < 8000:
        return 30
    else:
        return 50

df_test["yake_k"] = df_test["meta_len"].apply(decide_top_k)


In [270]:
import yake

kw_extractor = yake.KeywordExtractor(n=1, dedupLim=0.9, dedupFunc="seq")

def extract_yake(text, k):
    try:
        kws = kw_extractor.extract_keywords(text)
        return [kw for kw, score in kws[:k]]
    except:
        return []

df_test["yake_keywords"] = df_test.apply(
    lambda row: extract_yake(row["metadata_text_clean"], row["yake_k"]),
    axis=1
)


In [271]:
def yake_list_to_text(kws):
    if isinstance(kws, list):
        return " ".join(str(k).strip() for k in kws if isinstance(k, str))
    return ""

df_test["yake_text"] = df_test["yake_keywords"].apply(yake_list_to_text)


In [272]:
df_test.head(2)

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,metadata_text,metadata_text_clean,meta_len,yake_k,yake_keywords,yake_text
500081,Fujitsu,FSP:G-SW3Z560PRE0S,FSP:G-SW3Z560PRE0S,"SP 3y TS Sub & Upgr, 9x5, 4h Rm Rt f/ CS200c A...",Fujitsu FSP:G-SW3Z560PRE0S. Number of years: 3...,In times of growing complexity and decreasing ...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,Fujitsu FSP:G-SW3Z560PRE0S FSP:G-SW3Z560PRE0S ...,Fujitsu FSP:G-SW3Z560PRE0S FSP:G-SW3Z560PRE0S ...,1855,20,"[FSP, Fujitsu, Upgr, Advanced, EMEIA, support,...",FSP Fujitsu Upgr Advanced EMEIA support time S...
741063,HP,5KM83PA,13-ap0023tu,"Intel® Core™ i7-8565U (1.8 GHz), 16GB DDR4-SDR...",HP Spectre x360 13-ap0023tu. Product type: Hyb...,<b>Revolutionary battery life on a convertible...,Computers & Electronics,Computers,Notebooks,Computers & Electronics > Computers > Notebooks,HP 5KM83PA 13-ap0023tu Intel® Core™ i7-8565U (...,HP 5KM83PA 13-ap0023tu Intel® Core™ i7-8565U (...,1138,15,"[IPS, Touch, WLAN, Bluetooth, Core, Intel, UHD...",IPS Touch WLAN Bluetooth Core Intel UHD SSD Pr...


In [273]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-MiniLM-L6-v2")

test_emb = st_model.encode(
    df_test["yake_text"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [275]:
import numpy as np

# 1️⃣ Make sure you still have TRAIN embeddings (same order as df_clean)
# If you don't, recompute exactly like before:
# st_model = SentenceTransformer("all-MiniLM-L6-v2")
# embeddings = st_model.encode(
#     df_clean["yake_text"].fillna("").tolist(),
#     batch_size=64,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

train_emb = embeddings          # shape (N_train, 384)

# 2️⃣ Recompute C-level centroids in *embedding* space, NOT UMAP space
valid_cids = sorted(df_clean["C_id"].unique())
C_centroids = []
C_labels = []

for cid in valid_cids:
    idx = (df_clean["C_id"] == cid).to_numpy()
    C_centroids.append(train_emb[idx].mean(axis=0))   # mean in 384-d space
    C_labels.append(cid)

C_centroids = np.vstack(C_centroids)   # shape (K, 384)
C_labels = np.array(C_labels)

# 3️⃣ Nearest-centroid assignment for TEST (also 384-d, already normalized)
# test_emb you already computed from df_test["yake_text"]
similarity = np.dot(test_emb, C_centroids.T)   # (N_test, K)

test_pred_C_id_index = np.argmax(similarity, axis=1)
index_to_cid = {i: cid for i, cid in enumerate(C_labels)}

df_test["pred_C_id"] = [index_to_cid[i] for i in test_pred_C_id_index]

df_test[["pred_C_id"]].head()


,pred_C_id
500081,15
741063,78
1091454,60
522928,64
479478,55


In [276]:
df_test["pred_B_id"]   = df_test["pred_C_id"].map(C_to_B)
df_test["pred_A_id"]   = df_test["pred_B_id"].map(B_to_A)
df_test["pred_C_name"] = df_test["pred_C_id"].map(C_name_map)
df_test["pred_B_name"] = df_test["pred_B_id"].map(B_name_map)
df_test["pred_A_name"] = df_test["pred_A_id"].map(A_name_map)


In [277]:
df_test

,Brand,BrandPartCode,ProductName,Description.LongProductName,SummaryDescription.LongSummaryDescription,Description.LongDesc,A,B,C,path_3,...,meta_len,yake_k,yake_keywords,yake_text,pred_C_id,pred_B_id,pred_A_id,pred_C_name,pred_B_name,pred_A_name
500081,Fujitsu,FSP:G-SW3Z560PRE0S,FSP:G-SW3Z560PRE0S,"SP 3y TS Sub & Upgr, 9x5, 4h Rm Rt f/ CS200c A...",Fujitsu FSP:G-SW3Z560PRE0S. Number of years: 3...,In times of growing complexity and decreasing ...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,...,1855,20,"[FSP, Fujitsu, Upgr, Advanced, EMEIA, support,...",FSP Fujitsu Upgr Advanced EMEIA support time S...,15,22,38,Computer Server Support,Server Technical Support,Server Hardware Support
741063,HP,5KM83PA,13-ap0023tu,"Intel® Core™ i7-8565U (1.8 GHz), 16GB DDR4-SDR...",HP Spectre x360 13-ap0023tu. Product type: Hyb...,<b>Revolutionary battery life on a convertible...,Computers & Electronics,Computers,Notebooks,Computers & Electronics > Computers > Notebooks,...,1138,15,"[IPS, Touch, WLAN, Bluetooth, Core, Intel, UHD...",IPS Touch WLAN Bluetooth Core Intel UHD SSD Pr...,78,6,35,Computer Peripherals,Computer and Mobile,Electronics Equipment
1091454,HP,4NG33EA,880-156nf,None,HP OMEN 880-156nf. Processor frequency: 3.2 GH...,None,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,...,560,15,"[OMEN, Processor, Internal, model, memory, Int...",OMEN Processor Internal model memory Intel Cor...,60,12,0,Laptop Computer Hardware,Mobile Computing Hardware,Mobile Computing Hardware
522928,C2G,83061,1m Cat5e Non-Booted Unshielded (UTP) Network P...,Cat5E Assembled UTP Patch Cable Green 1m,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,Perfect for your home office or a large instal...,Computers & Electronics,Computer Cables,Networking Cables,Computers & Electronics > Computer Cables > Ne...,...,692,15,"[UTP, Unshielded, Patch, Green, Cable, Cables,...",UTP Unshielded Patch Green Cable Cables Assemb...,64,4,34,Networking Cabling Products,Computer Network Equipment,Network Infrastructure
479478,Lenovo,5PS0A14091,5PS0A14091,3YR Onsite + Keep Your Drive,"Lenovo 5PS0A14091. Number of years: 3 year(s),...",Lenovo offers a comprehensive portfolio of val...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,...,1900,20,"[Lenovo, Business, Service, Day, repair, Produ...",Lenovo Business Service Day repair Product Ons...,55,18,1,Computer Hardware Services,Computer Equipment Services,Computer and Technology
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859244,QNAP,TS-1635AX-4G/48TB-IWPRO,TS-1635AX-4G/48TB-IWPRO 16 Bay NAS,Marvell ARMADA 8040 ARMv8 Cortex-A72 quad-core...,QNAP TS-1635AX-4G/48TB-IWPRO 16 Bay NAS. Insta...,The TS-1635AX is powered by a Marvell® ARMADA®...,Computers & Electronics,Data Storage,Data Storage Devices,Computers & Electronics > Data Storage > Data ...,...,6006,30,"[NAS, storage, QNAP, data, expansion, Supporte...",NAS storage QNAP data expansion Supported driv...,20,14,3,Storage Network Devices,Storage Devices and,Storage and Organization
7400,Eaton,PW104BA0UA67,Powerware ePDU Basic 0U 16A C13 x 16,Powerware ePDU Basic 0U 16A C13 x 16,Eaton Powerware ePDU Basic 0U 16A C13 x 16. Ra...,Defining a ePDU<br>\r\n-Enclosure based Power ...,Computers & Electronics,Batteries & Power Supplies,Power Distribution Units (PDUs),Computers & Electronics > Batteries & Power Su...,...,1332,15,"[Powerware, Eaton, Basic, ePDU, Power, IEC, da...",Powerware Eaton Basic ePDU Power IEC data Rack...,31,21,41,Data Center Cabinets,Server Racks and,Computer Server Systems
895533,Lenovo,FRU04Y0249,04Y0249,Keyboard (Portuguese),Lenovo 04Y0249. Type: Keyboard. Keyboard langu...,,Computers & Electronics,Computers,Notebook Parts & Accessories,Computers & Electronics > Computers > Notebook...,...,186,10,"[Lenovo, Portuguese, 

In [278]:
# Gold labels from test
df_test["gold_A"]    = df_test["A"].fillna("").astype(str)
df_test["gold_B"]    = df_test["B"].fillna("").astype(str)
df_test["gold_C"]    = df_test["C"].fillna("").astype(str)
df_test["gold_path"] = df_test["path_3"].fillna("").astype(str)

# Predicted labels from your trained taxonomy
df_test["pred_A"] = df_test["pred_A_name"].fillna("").astype(str)
df_test["pred_B"] = df_test["pred_B_name"].fillna("").astype(str)
df_test["pred_C"] = df_test["pred_C_name"].fillna("").astype(str)

# Build predicted full path string (A > B > C)
df_test["pred_path"] = (
    df_test["pred_A"] + " > " +
    df_test["pred_B"] + " > " +
    df_test["pred_C"]
).str.replace(r"\s+>\s+$", "", regex=True)  # just in case of trailing " >"


In [279]:
df_test[["gold_A","gold_B","gold_C","gold_path",
         "pred_A","pred_B","pred_C","pred_path"]].head(10)


,gold_A,gold_B,gold_C,gold_path,pred_A,pred_B,pred_C,pred_path
500081,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,Server Hardware Support,Server Technical Support,Computer Server Support,Server Hardware Support > Server Technical Sup...
741063,Computers & Electronics,Computers,Notebooks,Computers & Electronics > Computers > Notebooks,Electronics Equipment,Computer and Mobile,Computer Peripherals,Electronics Equipment > Computer and Mobile > ...
1091454,Computers & Electronics,Computers,PCs/Workstations,Computers & Electronics > Computers > PCs/Work...,Mobile Computing Hardware,Mobile Computing Hardware,Laptop Computer Hardware,Mobile Computing Hardware > Mobile Computing H...
522928,Computers & Electronics,Computer Cables,Networking Cables,Computers & Electronics > Computer Cables > Ne...,Network Infrastructure,Computer Network Equipment,Networking Cabling Products,Network Infrastructure > Computer Network Equi...
479478,Computers & Electronics,Warranty & Support,Warranty & Support Extensions,Computers & Electronics > Warranty & Support >...,Computer and Technology,Computer Equipment Services,Computer Hardware Services,Computer and Technology > Computer Equipment S...
449642,Computers & Electronics,Data Storage,Data Storage Devices,Computers & Electronics > Data Storage > Data ...,Storage and Organization,Storage Devices and,External Hard Drives,Storage and Organization > Storage Devices and...
1024163,Computers & Electronics,Consumer Audio & Video Equipment,Audio Equipment Parts & Accessories,Computers & Electronics > Consumer Audio & Vid...,Ear Monitoring Systems,In Ear Monitors,Personal In Ear,Ear Monitoring Systems > In Ear Monitors > Per...
31904,Computers & Electronics,Printers & Scanners,Print & Scan Accessories,Computers & Electronics > Printers & Scanners ...,Office Supplies,Stationery and Office,Office Supplies,Office Supplies > Stationery and Office > Offi...
360523,Computers & Electronics,Telecom & Navigation,Mobile Phone Cases,Computers & Electronics > Telecom & Navigation...,Accessories for Phones,Phone Case Accessories,Tablet Computer Accessories,Accessories for Phones > Phone Case Accessorie...
675402,Computers & Electronics,Computers,Notebooks,Computers & Electronics > Computers > Notebooks,Computer Hardware Products,Electronics Computer,Electronics Computer Devices,Computer Hardware Products > Electronics Compu...


In [280]:
from sentence_transformers import SentenceTransformer
import numpy as np

st_eval = SentenceTransformer("all-MiniLM-L6-v2")

def mean_cosine_sim(refs, cands, model):
    refs  = [r if isinstance(r, str) else "" for r in refs]
    cands = [c if isinstance(c, str) else "" for c in cands]

    emb_ref  = model.encode(refs,  normalize_embeddings=True, convert_to_numpy=True)
    emb_cand = model.encode(cands, normalize_embeddings=True, convert_to_numpy=True)

    cos = np.sum(emb_ref * emb_cand, axis=1)   # dot product = cosine (L2-normalized)
    return float(cos.mean())

A_cos   = mean_cosine_sim(df_test["gold_A"],   df_test["pred_A"],   st_eval)
B_cos   = mean_cosine_sim(df_test["gold_B"],   df_test["pred_B"],   st_eval)
C_cos   = mean_cosine_sim(df_test["gold_C"],   df_test["pred_C"],   st_eval)
path_cos= mean_cosine_sim(df_test["gold_path"],df_test["pred_path"],st_eval)

print("TEST A-level cosine:",   A_cos)
print("TEST B-level cosine:",   B_cos)
print("TEST C-level cosine:",   C_cos)
print("TEST full-path cosine:", path_cos)


TEST A-level cosine: 0.47374793887138367
TEST B-level cosine: 0.47490885853767395
TEST C-level cosine: 0.46893489360809326
TEST full-path cosine: 0.5668718814849854


In [281]:
print("TEST A-level BERTScore:")
print(compute_bertscore(df_test["gold_A"].tolist(),
                        df_test["pred_A"].tolist()))

print("\nTEST B-level BERTScore:")
print(compute_bertscore(df_test["gold_B"].tolist(),
                        df_test["pred_B"].tolist()))

print("\nTEST C-level BERTScore:")
print(compute_bertscore(df_test["gold_C"].tolist(),
                        df_test["pred_C"].tolist()))

print("\nTEST Full-path BERTScore:")
print(compute_bertscore(df_test["gold_path"].tolist(),
                        df_test["pred_path"].tolist()))


TEST A-level BERTScore:
(0.5846969485282898, 0.5558198690414429, 0.5690906047821045)

TEST B-level BERTScore:
(0.5508848428726196, 0.6262550950050354, 0.5830051302909851)

TEST C-level BERTScore:
(0.5956989526748657, 0.5861158967018127, 0.5884203910827637)

TEST Full-path BERTScore:
(0.6752740740776062, 0.6738330125808716, 0.6738173365592957)


In [282]:
import pandas as pd

# Make sure these exist on df_test:
# A, B, C, path_3 (gold from Icecat)
# pred_A_name, pred_B_name, pred_C_name (from your nearest-centroid mapping)

df_test["gold_A"]   = df_test["A"].fillna("").astype(str)
df_test["gold_B"]   = df_test["B"].fillna("").astype(str)
df_test["gold_C"]   = df_test["C"].fillna("").astype(str)
df_test["gold_path"] = df_test["path_3"].fillna("").astype(str)

df_test["pred_A"]   = df_test["pred_A_name"].fillna("").astype(str)
df_test["pred_B"]   = df_test["pred_B_name"].fillna("").astype(str)
df_test["pred_C"]   = df_test["pred_C_name"].fillna("").astype(str)

df_test["pred_path"] = (
    "Computers & Electronics > "
    + df_test["pred_A"]
    + " > "
    + df_test["pred_B"]
    + " > "
    + df_test["pred_C"]
)


In [283]:
from sentence_transformers import SentenceTransformer
import numpy as np

st_eval = SentenceTransformer("all-MiniLM-L6-v2")

def mean_cosine(gold_list, pred_list):
    gold_emb = st_eval.encode(gold_list, normalize_embeddings=True)
    pred_emb = st_eval.encode(pred_list, normalize_embeddings=True)
    cos = np.sum(gold_emb * pred_emb, axis=1)
    return float(cos.mean())

gold_A   = df_test["gold_A"].tolist()
gold_B   = df_test["gold_B"].tolist()
gold_C   = df_test["gold_C"].tolist()
gold_path = df_test["gold_path"].tolist()

pred_A   = df_test["pred_A"].tolist()
pred_B   = df_test["pred_B"].tolist()
pred_C   = df_test["pred_C"].tolist()
pred_path = df_test["pred_path"].tolist()

print("TEST A-level cosine:",   mean_cosine(gold_A,   pred_A))
print("TEST B-level cosine:",   mean_cosine(gold_B,   pred_B))
print("TEST C-level cosine:",   mean_cosine(gold_C,   pred_C))
print("TEST full-path cosine:", mean_cosine(gold_path, pred_path))


TEST A-level cosine: 0.47374793887138367
TEST B-level cosine: 0.47490885853767395
TEST C-level cosine: 0.46893489360809326
TEST full-path cosine: 0.6354767680168152


In [284]:
print("TEST A-level BERTScore:")
print(compute_bertscore(gold_A, pred_A))

print("\nTEST B-level BERTScore:")
print(compute_bertscore(gold_B, pred_B))

print("\nTEST C-level BERTScore:")
print(compute_bertscore(gold_C, pred_C))

print("\nTEST Full-path BERTScore:")
print(compute_bertscore(gold_path, pred_path))


TEST A-level BERTScore:
(0.5846969485282898, 0.5558198690414429, 0.5690906047821045)

TEST B-level BERTScore:
(0.5508848428726196, 0.6262550950050354, 0.5830051302909851)

TEST C-level BERTScore:
(0.5956989526748657, 0.5861158967018127, 0.5884203910827637)

TEST Full-path BERTScore:
(0.7505324482917786, 0.8047842383384705, 0.775732159614563)
